# GroundTwin · Notebook 1 — Earth Radar Twin (RCM)
**Goal:** measure the radar *circular polarization ratio* (CPR) over Canadian places where we know the ground truth — glacier ice, rough rock, smooth sand/plains — using RADARSAT Constellation Mission (RCM) compact-pol ARD. This becomes the Earth reference ("ground truth") for reading NASA Mini-RF CPR maps of the lunar south pole.

**Как запускать:** Runtime → Run all (или по одной ячейке Shift+Enter). Ничего устанавливать на компьютер не нужно. Займёт ~10–20 минут.

Data: RCM CEOS-ARD, NRCan/CSA, open on AWS (`s3://rcm-ceos-ard`), discovered via the EODMS STAC API.
RADARSAT Constellation Mission Imagery © Government of Canada (2025). RADARSAT is an official mark of the Canadian Space Agency.

*Code written with assistance from Claude (Anthropic); site selection, checks and interpretation by Team Code Influence.*

In [ ]:
!pip -q install pystac-client rioxarray
import os, json, math, zipfile, warnings
import numpy as np, matplotlib.pyplot as plt
import rioxarray as rxr, pyproj
from shapely.geometry import box
from shapely.ops import transform
from pystac_client import Client
warnings.filterwarnings('ignore')
os.environ['GDAL_DISABLE_READDIR_ON_OPEN'] = 'EMPTY_DIR'   # faster cloud reads
os.environ['CPL_VSIL_CURL_ALLOWED_EXTENSIONS'] = '.tif'
OUT = '/content/groundtwin_out'; os.makedirs(f'{OUT}/site_maps', exist_ok=True)
print('ready')

## 1. Ground-truth sites
Each site = a small box (~3–5 km) where we know what the surface is. **Before trusting results, open each Google Maps link printed below and check the box really sits on ice / lava / sand.** Edit coordinates here if needed.

Classes: `ice` (glacier ice caps — dry firn/ice gives volume scattering), `rough_rock` (impact breccia, blocky lava — boulders give double bounces), `smooth` (sand dunes, flat plains — regolith-like smooth surfaces).

In [ ]:
# name: (class, lon, lat, half-size in degrees of latitude)
SITES = {
  'devon_ice_cap':      ('ice',        -82.00, 75.25, 0.02),
  'penny_ice_cap':      ('ice',        -66.00, 67.25, 0.02),
  'columbia_icefield':  ('ice',       -117.33, 52.17, 0.015),
  'haughton_crater':    ('rough_rock', -89.68, 75.38, 0.02),
  'nisgaa_lava_bed':    ('rough_rock',-128.97, 55.17, 0.015),
  'edziza_lava_field':  ('rough_rock',-130.65, 57.70, 0.015),
  'athabasca_dunes':    ('smooth',    -108.75, 59.08, 0.02),
  'great_sand_hills':   ('smooth',    -109.30, 50.45, 0.02),
  'regina_plain':       ('smooth',    -104.60, 50.55, 0.02),
}
DATE_RANGE = '2024-01-01/2026-11-13'
WINTER = {12, 1, 2, 3}          # dry snow / frozen ground preferred: less moisture effect on radar
MAX_TRIES = 6                   # scenes tried per site
INC_RANGE = (30, 60)            # keep pixels with local incidence angle in this range (Mini-RF ~48 deg)
LOOK = 5                        # 5x5 averaging -> ~100 m cells, reduces speckle

def site_bbox(lon, lat, h):
    dlon = h / max(math.cos(math.radians(lat)), 0.15)
    return (lon - dlon, lat - h, lon + dlon, lat + h)

for n, (cls, lon, lat, h) in SITES.items():
    print(f'{n:20s} {cls:11s} https://www.google.com/maps/@{lat},{lon},12z/data=!3m1!1e3')

## 2. Helper functions

In [ ]:
cat = Client.open('https://www.eodms-sgdot.nrcan-rncan.gc.ca/search/')

def read_clip(url, bbox):
    da = rxr.open_rasterio(url, masked=True)
    epsg = da.rio.crs.to_epsg()
    tr = pyproj.Transformer.from_crs(4326, epsg, always_xy=True)
    g = transform(tr.transform, box(*bbox))
    return da.rio.clip_box(*g.bounds).squeeze().astype('float64')

def block_mean(a, k):
    h, w = (a.shape[0] // k) * k, (a.shape[1] // k) * k
    if h == 0 or w == 0: return np.full((1, 1), np.nan)
    return np.nanmean(a[:h, :w].reshape(h // k, k, w // k, k), axis=(1, 3))

def find_items(bbox):
    s = cat.search(collections=['rcm-ard'], bbox=bbox, datetime=DATE_RANGE, limit=200, method='GET')
    items = [i for i in s.items() if 'rr' in i.assets and 'rl' in i.assets]
    items.sort(key=lambda i: i.properties['datetime'], reverse=True)       # newest first
    month = lambda i: int(i.properties['datetime'][5:7])
    return [i for i in items if month(i) in WINTER] + [i for i in items if month(i) not in WINTER]

def process_site(name, cls, lon, lat, h):
    bbox = site_bbox(lon, lat, h)
    items = find_items(bbox)
    print(f'{name}: {len(items)} scenes found')
    for it in items[:MAX_TRIES]:
        try:
            rr = read_clip(it.assets['rr'].href, bbox).values
            rl = read_clip(it.assets['rl'].href, bbox).values
        except Exception as e:
            print('   read failed:', str(e)[:80]); continue
        n0, n1 = min(rr.shape[0], rl.shape[0]), min(rr.shape[1], rl.shape[1])
        rr, rl = rr[:n0, :n1], rl[:n0, :n1]
        valid = np.isfinite(rr) & np.isfinite(rl) & (rr > 0) & (rl > 0)
        if valid.mean() < 0.6:
            print(f"   {it.properties['datetime'][:10]}: only {valid.mean():.0%} coverage, next"); continue
        inc_mean = None
        if 'local_inc_angle' in it.assets:
            try:
                inc = read_clip(it.assets['local_inc_angle'].href, bbox).values[:n0, :n1]
                inc_mean = float(np.nanmean(inc))
                valid &= (inc >= INC_RANGE[0]) & (inc <= INC_RANGE[1])
            except Exception: pass
        rr = np.where(valid, rr, np.nan); rl = np.where(valid, rl, np.nan)
        # ratio of local averages (like Mini-RF), not average of ratios
        cpr = block_mean(rr, LOOK) / block_mean(rl, LOOK)
        cpr = cpr[np.isfinite(cpr)]
        if cpr.size < 20:
            print('   too few valid cells, next'); continue
        print(f"   using {it.properties['datetime'][:10]}, {cpr.size} cells")
        return dict(site=name, cls=cls, lon=lon, lat=lat, date=it.properties['datetime'][:10],
                    scene=it.properties.get('order_key', it.id), inc_mean=inc_mean,
                    winter=int(it.properties['datetime'][5:7]) in WINTER,
                    n_cells=int(cpr.size), cpr_raw=cpr, map_rr=rr, map_rl=rl)
    print('   no usable scene'); return None
print('helpers ready')

## 3. Run all sites
⏳ Reads only the small boxes from the cloud, not whole scenes.

In [ ]:
results = []
for n, (cls, lon, lat, h) in SITES.items():
    r = process_site(n, cls, lon, lat, h)
    if r: results.append(r)
print(f'\nUsable sites: {len(results)} / {len(SITES)}')

## 4. Check the channel convention (important!)
CPR = same-sense / opposite-sense. For smooth natural surfaces CPR < 1 (single bounce dominates). We computed RR/RL. If the **smooth** sites come out > 1, RL is the same-sense channel and we flip the ratio. This cell decides and prints what it did — write the result in the paper's Methods.

In [ ]:
smooth_med = [np.median(r['cpr_raw']) for r in results if r['cls'] == 'smooth']
ref = np.median(smooth_med) if smooth_med else np.median(np.concatenate([r['cpr_raw'] for r in results]))
FLIP = bool(ref > 1)
for r in results:
    r['cpr'] = 1 / r['cpr_raw'] if FLIP else r['cpr_raw']
print('Reference median RR/RL =', round(float(ref), 3))
print('Convention used:', 'CPR = RL/RR (flipped)' if FLIP else 'CPR = RR/RL')

## 5. Results table + plots

In [ ]:
rows = []
for r in results:
    c = r['cpr']
    r['stats'] = dict(median=float(np.median(c)), p25=float(np.percentile(c, 25)),
                      p75=float(np.percentile(c, 75)), frac_gt1=float((c > 1).mean()))
    rows.append((r['cls'], r['site'], r['date'], r['winter'], r['inc_mean'], r['stats']['median'], r['stats']['frac_gt1'], r['n_cells']))
rows.sort()
print(f"{'class':11s} {'site':20s} {'date':10s} {'winter':6s} {'inc':>5s} {'CPRmed':>7s} {'>1':>5s} {'cells':>6s}")
for cls, s, d, w, inc, med, f1, n in rows:
    inc_s = '-' if inc is None else str(round(inc))
    print(f"{cls:11s} {s:20s} {d} {str(w):6s} {inc_s:>5s} {med:7.3f} {f1:5.0%} {n:6d}")

classes = ['smooth', 'rough_rock', 'ice']
colors = {'smooth': '#8C9BAB', 'rough_rock': '#D9822B', 'ice': '#4FB3D9'}
fig, ax = plt.subplots(figsize=(9, 5))
data, labels, cols = [], [], []
for cls in classes:
    for r in [r for r in results if r['cls'] == cls]:
        data.append(r['cpr']); labels.append(r['site'].replace('_', '\n')); cols.append(colors[cls])
bp = ax.boxplot(data, patch_artist=True, showfliers=False)
ax.set_xticks(range(1, len(labels) + 1)); ax.set_xticklabels(labels)
for p, c in zip(bp['boxes'], cols): p.set_facecolor(c)
ax.axhline(1, ls='--', c='k', lw=1); ax.set_ylabel('CPR (same-sense / opposite-sense)')
ax.set_title('Earth ground truth: RCM C-band CPR by surface class')
for cls in classes: ax.plot([], [], 's', c=colors[cls], label=cls)
ax.legend(); plt.xticks(fontsize=8); plt.tight_layout()
plt.savefig(f'{OUT}/earth_cpr_by_class.png', dpi=200); plt.show()

CPR maps for each site (same colour scale) — for the website and slides.

In [ ]:
for r in results:
    rr, rl = r['map_rr'], r['map_rl']
    m = block_mean(rl, LOOK) / block_mean(rr, LOOK) if FLIP else block_mean(rr, LOOK) / block_mean(rl, LOOK)
    plt.figure(figsize=(4, 4)); plt.imshow(m, cmap='magma', vmin=0, vmax=1.5)
    plt.colorbar(label='CPR', shrink=0.8); plt.title(f"{r['site']} ({r['cls']})\n{r['date']}"); plt.axis('off')
    plt.savefig(f"{OUT}/site_maps/{r['site']}.png", dpi=150, bbox_inches='tight'); plt.close()
print('saved', len(results), 'maps')

## 6. Export for the website and paper

In [ ]:
summary = dict(
  convention='CPR = RL/RR' if FLIP else 'CPR = RR/RL', look=LOOK, inc_range=list(INC_RANGE), date_range=DATE_RANGE,
  credit='RADARSAT Constellation Mission Imagery (c) Government of Canada (2025). RADARSAT is an official mark of the Canadian Space Agency.',
  sites=[dict(site=r['site'], cls=r['cls'], lon=r['lon'], lat=r['lat'], date=r['date'], scene=r['scene'],
              winter=r['winter'], inc_mean=r['inc_mean'], n_cells=r['n_cells'], **r['stats'],
              hist=np.histogram(np.clip(r['cpr'], 0, 2), bins=40, range=(0, 2))[0].tolist())
         for r in results])
cls_summary = {}
for cls in classes:
    sel = [r['cpr'] for r in results if r['cls'] == cls]
    if sel:
        allc = np.concatenate(sel)
        cls_summary[cls] = dict(median=float(np.median(allc)), p25=float(np.percentile(allc, 25)),
                                p75=float(np.percentile(allc, 75)), frac_gt1=float((allc > 1).mean()), n=int(allc.size))
summary['classes'] = cls_summary
json.dump(summary, open(f'{OUT}/earth_cpr.json', 'w'), indent=1)
print(json.dumps(cls_summary, indent=1))

with zipfile.ZipFile('/content/groundtwin_earth_radar.zip', 'w') as z:
    for root, _, fs in os.walk(OUT):
        for f in fs: z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), OUT))
from google.colab import files; files.download('/content/groundtwin_earth_radar.zip')

## What to send back to Claude
1. The **results table** from section 5 (copy the text).
2. The **convention** line from section 4.
3. The boxplot picture (screenshot is fine).

**What we hope to see:** smooth < rough_rock, and ice highest (often > 1 for dry firn — the same "anomalous" signature reported for Greenland's ice sheet and icy moons). If ice is *not* higher than rock, that is also a real, honest result — it shows why CPR alone can't prove lunar ice.